# Predictive Maintenance of Industrial Machinery
## AICTE-2026 Problem Statement #39

**Objective:** Build a machine learning classification system that predicts whether industrial  
machinery is operating normally or experiencing a failure, and — where the data supports it —  
identifies the failure type.

**Dataset:** Machine Predictive Maintenance Classification  
Source: [Kaggle — Shivam Bansal](https://www.kaggle.com/datasets/shivamb/machine-predictive-maintenance-classification)  

---

### Notebook Structure
| Section | Title |
|---------|-------|
| 0 | Setup & Imports |
| 1 | Data Loading |
| 2 | Data Cleaning & Validation |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Preprocessing & Feature Engineering |
| 5 | Model Training & Comparison |
| 6 | Best Model Evaluation |
| 7 | Failure Type Classifier (Secondary) |
| 8 | Save Models |
| 9 | Sample Predictions |

> **Note:** All metrics, plots, and statistics in this notebook are computed from the  
> actual dataset. Nothing is pre-filled or fabricated.

---
## Section 0 — Setup & Imports

In [ ]:
# ── Standard library ────────────────────────────────────────
import warnings
from pathlib import Path

# ── Data science ─────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn — preprocessing ──────────────────────────────
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# ── Scikit-learn — models ─────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# ── Scikit-learn — metrics ─────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

# ── Model persistence ─────────────────────────────────────────
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
print('All imports successful.')

In [ ]:
# ── Path constants ─────────────────────────────────────────────
# Notebook is at the project root, so '.' is the project root.
PROJECT_ROOT = Path('.')
DATA_DIR     = PROJECT_ROOT / 'data'
MODEL_DIR    = PROJECT_ROOT / 'model'

RAW_CSV           = DATA_DIR  / 'predictive_maintenance.csv'
MODEL_PKL         = MODEL_DIR / 'best_model.pkl'
LABEL_ENCODER_PKL = MODEL_DIR / 'label_encoder.pkl'

# ── Feature / target column names ──────────────────────────────
DROP_COLUMNS        = ['UDI', 'Product ID']
CATEGORICAL_FEATURE = 'Type'
TYPE_CATEGORIES     = [['L', 'M', 'H']]
NUMERIC_FEATURES    = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
]
FEATURE_COLUMNS  = [CATEGORICAL_FEATURE] + NUMERIC_FEATURES
TARGET_BINARY    = 'Target'
TARGET_FAIL_TYPE = 'Failure Type'

# ── Reproducibility ────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20

print(f'Project root : {PROJECT_ROOT.resolve()}')
print(f'Dataset path : {RAW_CSV}')
print(f'Dataset exists: {RAW_CSV.exists()}')

---
## Section 1 — Data Loading

Load the raw CSV (tab-separated) and take a first look at its shape, columns, data types,  
and summary statistics. No modifications are made to the data in this section.

In [ ]:
# ── Load raw dataset ──────────────────────────────────────────
# The file is TAB-separated (verified during Step 2 dataset check).
df_raw = pd.read_csv(RAW_CSV, sep='\t')

print(f'Shape  : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head()

In [ ]:
# ── Data types and non-null counts ────────────────────────────
print('--- dtypes ---')
print(df_raw.dtypes)
print()

# ── Summary statistics for numeric columns ────────────────────
print('--- describe (numeric columns) ---')
df_raw.describe().round(4)

---
## Section 2 — Data Cleaning & Validation

Checks performed (all results computed from the actual dataset):
1. Null / missing values per column
2. Duplicate rows
3. `Target` ↔ `Failure Type` consistency audit  
   Known dataset quirks (published in the original Kaggle dataset):
   - 18 rows: `Target == 0` with `Failure Type == 'Random Failures'`  
     (random failures that did not trip the binary target — kept as-is)
   - 9 rows: `Target == 1` with `Failure Type == 'No Failure'`  
     (binary failure without a named type — kept as-is)
   These rows are **retained**; removing them would alter the published dataset.
4. Drop `UDI` and `Product ID` — sequential ID and serial number, no predictive value
5. Confirm the working DataFrame `df` is ready for EDA

In [ ]:
# ── 1. Null / missing values ──────────────────────────────────
null_counts = df_raw.isnull().sum()
print('Null values per column:')
print(null_counts)
print(f'\nTotal null cells: {null_counts.sum()}')
assert null_counts.sum() == 0, 'UNEXPECTED: null values found — investigate before proceeding'

In [ ]:
# ── 2. Duplicate rows ─────────────────────────────────────────
n_duplicates = df_raw.duplicated().sum()
print(f'Duplicate rows: {n_duplicates}')
assert n_duplicates == 0, 'UNEXPECTED: duplicate rows found — investigate before proceeding'

In [ ]:
# ── 3. Target ↔ Failure Type consistency audit ────────────────
#
# This is a REPORTING check, not an assertion — the inconsistencies
# below are known quirks in the published Kaggle dataset and are retained.
#
# Rule A: Target == 0  →  Failure Type should be 'No Failure'
rule_a = df_raw[
    (df_raw['Target'] == 0) & (df_raw['Failure Type'] != 'No Failure')
]
# Rule B: Target == 1  →  Failure Type should NOT be 'No Failure'
rule_b = df_raw[
    (df_raw['Target'] == 1) & (df_raw['Failure Type'] == 'No Failure')
]

print(f'Rule A (Target=0, Failure Type != No Failure): {len(rule_a)} rows')
if len(rule_a) > 0:
    print(rule_a[['Target', 'Failure Type']].value_counts().to_string())
    print('  → Known quirk: Random Failures with Target=0. Rows retained.')

print()
print(f'Rule B (Target=1, Failure Type == No Failure): {len(rule_b)} rows')
if len(rule_b) > 0:
    print(rule_b[['Target', 'Failure Type']].value_counts().to_string())
    print('  → Known quirk: Binary failure with no named type. Rows retained.')

print()
print('Consistency audit complete. Dataset used as published (no rows removed).')

In [ ]:
# ── 4. Drop non-predictive columns ───────────────────────────
# UDI: sequential row counter (1–10000) — no signal
# Product ID: alphanumeric serial — no signal
df = df_raw.drop(columns=DROP_COLUMNS)

print(f'Columns after drop: {df.columns.tolist()}')
print(f'Shape after drop  : {df.shape}')

In [ ]:
# ── 5. Confirm working DataFrame ─────────────────────────────
print('Working DataFrame df — summary')
print('=' * 45)
print(f'  Shape            : {df.shape}')
print(f'  Feature columns  : {FEATURE_COLUMNS}')
print(f'  Primary target   : {TARGET_BINARY!r}  →  unique values: {sorted(df[TARGET_BINARY].unique())}')
print(f'  Secondary target : {TARGET_FAIL_TYPE!r}  →  unique values:')
for val, cnt in df[TARGET_FAIL_TYPE].value_counts().items():
    print(f'      {val!r:35s}  {cnt:5d}')
print()
print('Section 2 complete. df is ready for EDA.')
df.head()

---
## Section 3 — Exploratory Data Analysis (EDA)

All plots are produced from the cleaned `df` DataFrame (10 000 rows, 8 columns).  
No values are pre-filled or fabricated.

**Plots in this section:**
1. Target class distribution (bar chart + percentages)
2. Failure Type distribution (bar chart)
3. Machine Type distribution (bar chart)
4. Numeric feature distributions split by Target (KDE + histograms)
5. Correlation heatmap of numeric features
6. Box plots of each numeric feature by Target

> **Dataset note:** The published Kaggle dataset contains 18 rows where `Target=0` but  
> `Failure Type='Random Failures'`, and 9 rows where `Target=1` but `Failure Type='No Failure'`.  
> These are known labelling quirks in the original data. All 10 000 rows are retained as published.

In [ ]:
# ── Plot 1: Target class distribution ────────────────────────
target_counts = df[TARGET_BINARY].value_counts().sort_index()
target_labels = {0: 'No Failure (0)', 1: 'Failure (1)'}
labels = [target_labels[i] for i in target_counts.index]
colors = ['#4C9BE8', '#E85C5C']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(labels, target_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
for bar, count in zip(bars, target_counts.values):
    pct = count / len(df) * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_ylim(0, max(target_counts.values) * 1.18)

# Pie chart
axes[1].pie(target_counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Target Class Share', fontsize=13, fontweight='bold')

plt.suptitle('Class Imbalance: ~96.6% No Failure vs ~3.4% Failure',
             fontsize=11, color='#555555', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Failure Type distribution ────────────────────────
ft_counts = df[TARGET_FAIL_TYPE].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All failure types (including No Failure)
palette_all = sns.color_palette('muted', len(ft_counts))
bars = axes[0].barh(ft_counts.index, ft_counts.values, color=palette_all, edgecolor='white')
axes[0].set_title('All Failure Type Counts', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()
for bar, count in zip(bars, ft_counts.values):
    axes[0].text(bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
                 f'{count:,} ({count/len(df)*100:.2f}%)', va='center', fontsize=9)
axes[0].set_xlim(0, ft_counts.max() * 1.25)

# Failure types only (exclude No Failure) for closer look
ft_failures = ft_counts[ft_counts.index != 'No Failure'].sort_values(ascending=True)
palette_fail = sns.color_palette('Set2', len(ft_failures))
bars2 = axes[1].barh(ft_failures.index, ft_failures.values, color=palette_fail, edgecolor='white')
axes[1].set_title('Failure Types Only (excluding No Failure)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')
for bar, count in zip(bars2, ft_failures.values):
    axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 f'{count} ({count/len(df)*100:.2f}%)', va='center', fontsize=9)
axes[1].set_xlim(0, ft_failures.max() * 1.35)

plt.tight_layout()
plt.show()

print('\nNote: 9 rows have Target=1 but Failure Type="No Failure" (known dataset quirk, retained).')
print('      18 rows have Target=0 but Failure Type="Random Failures" (known dataset quirk, retained).')

In [ ]:
# ── Plot 3: Machine Type distribution ────────────────────────
type_counts = df[CATEGORICAL_FEATURE].value_counts().sort_index()
type_colors = ['#7C9ED9', '#E8A55C', '#6DBFB0']  # H, L, M

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bars = axes[0].bar(type_counts.index, type_counts.values, color=type_colors, edgecolor='white')
axes[0].set_title('Machine Type Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Type (H=High, L=Low, M=Medium quality)')
axes[0].set_ylabel('Count')
for bar, (t, cnt) in zip(bars, type_counts.items()):
    pct = cnt / len(df) * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                 f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_ylim(0, type_counts.max() * 1.2)

# Failure rate per type
fail_rate_by_type = df.groupby(CATEGORICAL_FEATURE)[TARGET_BINARY].mean() * 100
fail_rate_by_type = fail_rate_by_type.loc[type_counts.index]
bars2 = axes[1].bar(fail_rate_by_type.index, fail_rate_by_type.values, color=type_colors, edgecolor='white')
axes[1].set_title('Failure Rate (%) by Machine Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Type')
axes[1].set_ylabel('Failure Rate (%)')
for bar, (t, rate) in zip(bars2, fail_rate_by_type.items()):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f'{rate:.2f}%', ha='center', va='bottom', fontsize=10)
axes[1].set_ylim(0, fail_rate_by_type.max() * 1.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Numeric feature distributions split by Target ─────
# Histogram + seaborn KDE overlays for each of the 5 sensor features.
# Blue = No Failure (Target=0), Red = Failure (Target=1).
# Uses seaborn.kdeplot (no scipy dependency required).

feature_units = {
    'Air temperature [K]'      : 'Kelvin',
    'Process temperature [K]'  : 'Kelvin',
    'Rotational speed [rpm]'   : 'RPM',
    'Torque [Nm]'              : 'Newton-metres',
    'Tool wear [min]'          : 'Minutes',
}

df_no_fail = df[df[TARGET_BINARY] == 0]
df_fail    = df[df[TARGET_BINARY] == 1]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(NUMERIC_FEATURES):
    ax = axes[i]
    ax.hist(df_no_fail[feat], bins=40, color='#4C9BE8', alpha=0.40,
            density=True, label='No Failure (0)')
    ax.hist(df_fail[feat],    bins=40, color='#E85C5C', alpha=0.50,
            density=True, label='Failure (1)')
    # KDE overlay via seaborn (no scipy needed)
    sns.kdeplot(data=df_no_fail, x=feat, ax=ax,
                color='#1A5FA8', linewidth=1.8, warn_singular=False)
    sns.kdeplot(data=df_fail,    x=feat, ax=ax,
                color='#A81A1A', linewidth=1.8, warn_singular=False)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel(feature_units[feat])
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

# Hide the unused 6th subplot slot
axes[5].set_visible(False)

plt.suptitle('Feature Distributions: No Failure vs Failure', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 5: Correlation heatmap of numeric features ───────────
# Include Target so we can see which features correlate with failure.
corr_cols = NUMERIC_FEATURES + [TARGET_BINARY]
corr_matrix = df[corr_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True   # hide upper triangle (redundant)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 9},
)
ax.set_title('Correlation Heatmap — Numeric Features + Target', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

print('\nTop correlations with Target (absolute value):')
target_corr = corr_matrix['Target'].drop('Target').abs().sort_values(ascending=False)
for feat, val in target_corr.items():
    print(f'  {feat:35s}: {val:.3f}')

In [ ]:
# ── Plot 6: Box plots of numeric features by Target ───────────
# Shows median, IQR, and outliers for each feature, split by failure status.

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

target_map     = {0: 'No Failure (0)', 1: 'Failure (1)'}
target_palette = {'No Failure (0)': '#4C9BE8', 'Failure (1)': '#E85C5C'}

# Temporary column with readable label for the legend
df_plot = df.copy()
df_plot['Failure Status'] = df_plot[TARGET_BINARY].map(target_map)

for i, feat in enumerate(NUMERIC_FEATURES):
    ax = axes[i]
    sns.boxplot(
        data=df_plot,
        x='Failure Status',
        y=feat,
        hue='Failure Status',
        palette=target_palette,
        legend=False,
        flierprops={'marker': 'o', 'markerfacecolor': 'grey',
                    'markersize': 2, 'alpha': 0.4},
        ax=ax,
    )
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(feature_units[feat])
    # Add median annotations
    for j, status in enumerate(['No Failure (0)', 'Failure (1)']):
        subset = df_plot[df_plot['Failure Status'] == status][feat]
        median_val = subset.median()
        ax.text(j, median_val, f' {median_val:.1f}', va='center',
                fontsize=8, color='black', fontweight='bold')

axes[5].set_visible(False)

plt.suptitle('Box Plots: Numeric Features by Failure Status', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Section 4 — Preprocessing & Feature Engineering

**Steps in this section:**
1. Define feature matrix `X` and binary target vector `y`
2. Build a `ColumnTransformer` preprocessor:
   - `OneHotEncoder(handle_unknown='ignore')` for `Type` (H / L / M — nominal, no ordering)
   - `StandardScaler` for the five numeric sensor features
3. Stratified 80/20 train/test split (`random_state=42`, `stratify=y`)
4. Wrap preprocessor in a `Pipeline` scaffold ready to receive a classifier in Section 5

> **Why OneHotEncoder, not OrdinalEncoder?**  
> `Type` values H, M, L represent *quality variants*, not an ordered scale.  
> OrdinalEncoder would impose a numeric ordering (e.g. H=2 > M=1 > L=0) that  
> tree and linear models would misinterpret as a meaningful magnitude difference.  
> OneHotEncoder creates three binary indicator columns and makes no ordering assumption.

> **Fit only on training data:** The preprocessor is fit inside the Pipeline using  
> `pipeline.fit(X_train, y_train)` (done in Section 5). It is *not* fit here on all of `X`  
> to avoid data leakage from test statistics into training.

In [ ]:
# ── 1. Define feature matrix X and target vector y ───────────
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_BINARY].copy()

print(f'X shape : {X.shape}  (rows x features)')
print(f'y shape : {y.shape}')
print()
print('Feature columns:')
for i, col in enumerate(X.columns):
    print(f'  [{i}] {col!r}  dtype={X[col].dtype}')
print()
print(f'Target distribution:')
for val, cnt in y.value_counts().sort_index().items():
    print(f'  y={val}  count={cnt:,}  ({cnt/len(y)*100:.2f}%)')

In [ ]:
# ── 2. Build the ColumnTransformer preprocessor ──────────────
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            [CATEGORICAL_FEATURE],          # ['Type']
        ),
        (
            'num',
            StandardScaler(),
            NUMERIC_FEATURES,               # 5 sensor columns
        ),
    ],
    remainder='drop',                       # drop any other columns (safety net)
    verbose_feature_names_out=True,
)

print('ColumnTransformer built:')
print(f'  Categorical transformer : OneHotEncoder  on {[CATEGORICAL_FEATURE]}')
print(f'  Numeric transformer     : StandardScaler on {NUMERIC_FEATURES}')
print(f'  remainder               : drop')

In [ ]:
# ── 3. Stratified train/test split ───────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,          # preserves class ratio in both splits
)

print('Train / test split (stratified):')
print(f'  X_train : {X_train.shape}   X_test : {X_test.shape}')
print(f'  y_train : {y_train.shape}   y_test : {y_test.shape}')
print()
print('Class distribution after split:')
for split_name, split_y in [('y_train', y_train), ('y_test', y_test)]:
    vc = split_y.value_counts().sort_index()
    parts = [f'  {split_name}: ' + '  '.join(
        f'class {v}={cnt:,} ({cnt/len(split_y)*100:.2f}%)'
        for v, cnt in vc.items()
    )]
    print(parts[0])

In [ ]:
# ── 4. Build Pipeline scaffold ───────────────────────────────
# The classifier step is a placeholder (None) until Section 5.
# The preprocessor is NOT fit here — it will be fit on X_train
# inside pipeline.fit() in Section 5 to prevent data leakage.

pipeline_scaffold = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   None),           # replaced with actual model in Section 5
])

# Smoke-test: fit the preprocessor alone on training data and
# inspect the output feature names to confirm OHE expansion.
preprocessor.fit(X_train)
encoded_feature_names = preprocessor.get_feature_names_out()

print('Pipeline scaffold created.')
print(f'  Steps   : {[name for name, _ in pipeline_scaffold.steps]}')
print()
print(f'Encoded feature names after preprocessing ({len(encoded_feature_names)} total):')
for name in encoded_feature_names:
    print(f'  {name}')
print()
print('Section 4 complete. Ready for Section 5 (model training).')

---
## Section 5 — Model Training & Comparison

**Strategy:** 5-fold stratified cross-validation on **training data only** — the test set is  
never touched during model selection. The model with the highest mean **macro F1** across  
the CV folds is selected. Raw accuracy is reported but not used for selection because the  
dataset is class-imbalanced (~3.4% failure rate).

| Model | Class-imbalance handling |
|-------|-------------------------|
| Logistic Regression | `class_weight='balanced'` |
| Decision Tree | `class_weight='balanced'` |
| Random Forest | `class_weight='balanced'` |
| Gradient Boosting | learns iteratively; no direct `class_weight`, compensated by CV |
| K-Nearest Neighbours | no native `class_weight`; included as a non-parametric baseline |

Each candidate is wrapped in a **full Pipeline** (preprocessor + classifier) so preprocessing  
is re-fit from scratch on each CV fold — no leakage.

In [ ]:
# ── Candidate models ─────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
import time

candidates = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ),
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        random_state=RANDOM_STATE
    ),
    'K-Nearest Neighbours': KNeighborsClassifier(
        n_neighbors=7, n_jobs=-1
    ),
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scoring   = ['f1_macro', 'roc_auc', 'accuracy',
                'precision_macro', 'recall_macro']

cv_results = {}
print('Running 5-fold stratified cross-validation on training data...')
print('(This may take ~1-2 minutes)')
print()

for name, clf in candidates.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('classifier', clf)])
    t0   = time.time()
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv_strategy,
        scoring=cv_scoring,
        return_train_score=False,
        n_jobs=1,
    )
    elapsed = time.time() - t0
    cv_results[name] = {
        'F1-macro (mean)' : scores['test_f1_macro'].mean(),
        'F1-macro (std)'  : scores['test_f1_macro'].std(),
        'ROC-AUC (mean)'  : scores['test_roc_auc'].mean(),
        'Accuracy (mean)' : scores['test_accuracy'].mean(),
        'Precision-macro' : scores['test_precision_macro'].mean(),
        'Recall-macro'    : scores['test_recall_macro'].mean(),
        'Time (s)'        : round(elapsed, 1),
    }
    print(f'  {name:25s}  F1-macro={scores["test_f1_macro"].mean():.4f}'
          f'  ROC-AUC={scores["test_roc_auc"].mean():.4f}'
          f'  [{elapsed:.1f}s]')

print('\nCross-validation complete.')

In [ ]:
# ── Comparison table ─────────────────────────────────────────
results_df = pd.DataFrame(cv_results).T.sort_values('F1-macro (mean)', ascending=False)
results_df = results_df.round(4)

print('Cross-validation results (sorted by F1-macro, highest first):')
print('NOTE: Raw accuracy is shown for reference only — it is NOT the selection criterion.')
print()
display(results_df)

# ── Select best model ─────────────────────────────────────────
best_model_name = results_df.index[0]
best_f1         = results_df.loc[best_model_name, 'F1-macro (mean)']
best_auc        = results_df.loc[best_model_name, 'ROC-AUC (mean)']

print(f'\nSelected model : {best_model_name!r}')
print(f'  CV F1-macro  : {best_f1:.4f}')
print(f'  CV ROC-AUC   : {best_auc:.4f}')
print(f'  Reason       : Highest mean macro F1 across 5 stratified CV folds.')

---
## Section 6 — Best Model: Final Evaluation & Saving

The selected model is now fit on the **full training set** and evaluated **once** on  
the untouched test set. All metric values below come from the actual test predictions.

In [ ]:
# ── Fit final pipeline on full training set ──────────────────
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay,
    f1_score, roc_auc_score, precision_score, recall_score,
)
from sklearn.inspection import permutation_importance

best_clf = candidates[best_model_name]

best_pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('classifier',   best_clf),
])
best_pipeline.fit(X_train, y_train)

print(f'Final pipeline fitted on X_train {X_train.shape}.')
print(f'  Model : {best_model_name}')
print(f'  Steps : {[s for s, _ in best_pipeline.steps]}')

In [ ]:
# ── Evaluate on test set (run once — never used for selection) ─
y_pred      = best_pipeline.predict(X_test)
y_pred_prob = best_pipeline.predict_proba(X_test)[:, 1]

test_f1       = f1_score(y_test, y_pred, average='macro')
test_auc      = roc_auc_score(y_test, y_pred_prob)
test_precision= precision_score(y_test, y_pred, average='macro')
test_recall   = recall_score(y_test, y_pred, average='macro')

print(f'Test-set evaluation  (model: {best_model_name})')
print('=' * 52)
print(f'  F1-macro    : {test_f1:.4f}')
print(f'  ROC-AUC     : {test_auc:.4f}')
print(f'  Precision   : {test_precision:.4f}  (macro)')
print(f'  Recall      : {test_recall:.4f}  (macro)')
print()
print('Classification report:')
print(classification_report(y_test, y_pred,
      target_names=['No Failure (0)', 'Failure (1)']))

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No Failure (0)', 'Failure (1)'],
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(
    f'Confusion Matrix — {best_model_name}\n'
    f'Test set  |  F1-macro={test_f1:.4f}  ROC-AUC={test_auc:.4f}',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print(f'False-negative rate (missed failures): {fn/(fn+tp)*100:.1f}%')

In [ ]:
# ── ROC curve ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(
    y_test, y_pred_prob,
    name=best_model_name,
    ax=ax,
    color='#3b82d4',
)
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random classifier')
ax.set_title(f'ROC Curve — {best_model_name}', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance ────────────────────────────────────────
# Use native feature_importances_ if the classifier exposes it
# (tree-based models); fall back to permutation importance otherwise.

encoded_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()
clf_step      = best_pipeline.named_steps['classifier']

if hasattr(clf_step, 'feature_importances_'):
    importances = clf_step.feature_importances_
    importance_method = 'Native feature_importances_'
else:
    # Permutation importance on the preprocessed test data
    X_test_transformed = best_pipeline.named_steps['preprocessor'].transform(X_test)
    perm = permutation_importance(
        clf_step, X_test_transformed, y_test,
        n_repeats=10, random_state=RANDOM_STATE, scoring='f1_macro',
    )
    importances = perm.importances_mean
    importance_method = 'Permutation importance (10 repeats, f1_macro)'

imp_df = pd.DataFrame({
    'Feature'   : encoded_names,
    'Importance': importances,
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#E85C5C' if v >= 0 else '#AAAAAA' for v in imp_df['Importance']]
ax.barh(imp_df['Feature'], imp_df['Importance'], color=colors, edgecolor='white')
ax.set_title(
    f'Feature Importance — {best_model_name}\n({importance_method})',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel('Importance')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(f'\nMethod: {importance_method}')
print(imp_df.sort_values('Importance', ascending=False).to_string(index=False))

In [ ]:
# ── Save and verify the model ─────────────────────────────────
import joblib

joblib.dump(best_pipeline, MODEL_PKL)
print(f'Model saved : {MODEL_PKL}')
print(f'File size   : {MODEL_PKL.stat().st_size / 1024:.1f} KB')

# ── Reload and verify ─────────────────────────────────────────
loaded_pipeline = joblib.load(MODEL_PKL)
y_pred_verify   = loaded_pipeline.predict(X_test)

assert (y_pred_verify == y_pred).all(), 'RELOAD CHECK FAILED: predictions differ!'
print()
print('Reload verification: PASS')
print(f'  Loaded pipeline type : {type(loaded_pipeline).__name__}')
print(f'  Steps                : {[s for s, _ in loaded_pipeline.steps]}')
print(f'  Prediction check     : all {len(y_pred)} test predictions match')
print()
print('Section 6 complete.')
print(f'  best_model.pkl saved to: {MODEL_PKL.resolve()}')

---
## Section 7 — Failure Type Classifier (Secondary)

A second, **independent** `sklearn.Pipeline` is trained to predict the specific  
failure type given that a failure has already been detected.  

**Input:** rows where `Target == 1` AND `Failure Type != 'No Failure'`  
**Target:** `Failure Type` (multiclass string label)  
**Features:** same six columns used by the primary binary classifier  
**Preprocessing:** `OneHotEncoder` for `Type`, `StandardScaler` for 5 numeric features  

| Failure Type | Samples (actual) |
|--------------|------------------|
| Heat Dissipation Failure | 112 |
| Power Failure | 95 |
| Overstrain Failure | 78 |
| Tool Wear Failure | 45 |

> **Note on 'Random Failures':** In this dataset, all 18 `'Random Failures'` rows have  
> `Target == 0`, so they are absent from the failure-only subset.  
> This class cannot be predicted by the secondary classifier and is excluded by design.

> **Note on 9 quirk rows:** The 9 rows with `Target == 1` and `Failure Type == 'No Failure'`  
> have no named type to predict and are excluded from this classifier.

> **Statistical caution:** Total sample size is 330. The smallest class (Tool Wear Failure,  
> n=45) gives ~9 test samples. Per-class metrics for small classes have high variance.

In [ ]:
# ── 1. Build the failure-type subset ─────────────────────────
# Keep only rows where Target==1 AND Failure Type is a named type.
# Excludes: 9 quirk rows (Target=1, Failure Type='No Failure')
# 'Random Failures' rows all have Target=0 so they are already absent.

df_failures = df[
    (df[TARGET_BINARY] == 1) &
    (df[TARGET_FAIL_TYPE] != 'No Failure')
].copy()

X_ft = df_failures[FEATURE_COLUMNS].copy()
y_ft = df_failures[TARGET_FAIL_TYPE].copy()

print('Failure-type subset:')
print(f'  Rows     : {len(df_failures)}')
print(f'  Features : {FEATURE_COLUMNS}')
print()
print('Class distribution:')
for cls, cnt in y_ft.value_counts().items():
    print(f'  {cls!r:35s}: {cnt}')
print()
print('NOTE: "Random Failures" is absent — all 18 such rows have Target=0 in this dataset.')
print('NOTE: 9 quirk rows (Target=1, Failure Type=No Failure) excluded — no type to predict.')

In [ ]:
# ── 2. Build pipeline, CV comparison, select best ────────────
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import time

# Stratified 80/20 split on the failure subset
X_ft_train, X_ft_test, y_ft_train, y_ft_test = train_test_split(
    X_ft, y_ft,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_ft,
)
print(f'Failure-type split: train={len(X_ft_train)}  test={len(X_ft_test)}')
print()

# Preprocessing: same structure as primary model
def build_ft_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [CATEGORICAL_FEATURE]),
            ('num', StandardScaler(), NUMERIC_FEATURES),
        ],
        remainder='drop',
        verbose_feature_names_out=True,
    )

ft_candidates = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3,
        random_state=RANDOM_STATE),
}

cv_ft = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
ft_cv_results = {}

print('5-fold stratified CV on failure-type training subset:')
for name, clf in ft_candidates.items():
    pipe = Pipeline([('preprocessor', build_ft_preprocessor()), ('classifier', clf)])
    t0   = time.time()
    scores = cross_validate(
        pipe, X_ft_train, y_ft_train,
        cv=cv_ft,
        scoring=['f1_macro', 'accuracy'],
        return_train_score=False,
        n_jobs=1,
    )
    elapsed = time.time() - t0
    ft_cv_results[name] = {
        'F1-macro (mean)': scores['test_f1_macro'].mean(),
        'F1-macro (std)':  scores['test_f1_macro'].std(),
        'Accuracy (mean)': scores['test_accuracy'].mean(),
        'Time (s)':        round(elapsed, 1),
    }
    print(f'  {name:25s}  F1-macro={scores["test_f1_macro"].mean():.4f}  [{elapsed:.1f}s]')

ft_results_df = pd.DataFrame(ft_cv_results).T.sort_values('F1-macro (mean)', ascending=False)
best_ft_name  = ft_results_df.index[0]
print()
display(ft_results_df.round(4))
print(f'\nSelected failure-type model: {best_ft_name!r}')
print(f'  CV F1-macro: {ft_results_df.loc[best_ft_name, "F1-macro (mean)"]:.4f}')

In [ ]:
# ── 3. Fit on full training subset and evaluate on test ───────
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

best_ft_pipeline = Pipeline([
    ('preprocessor', build_ft_preprocessor()),
    ('classifier',   ft_candidates[best_ft_name]),
])
best_ft_pipeline.fit(X_ft_train, y_ft_train)

y_ft_pred = best_ft_pipeline.predict(X_ft_test)

# Ordered class list for consistent display
ft_classes = sorted(y_ft.unique())

print(f'Failure-type test-set evaluation  (model: {best_ft_name})')
print('=' * 55)
print()
print('Classification report:')
print(classification_report(
    y_ft_test, y_ft_pred,
    labels=ft_classes,
    target_names=ft_classes,
    zero_division=0,
))
print('CAUTION: Tool Wear Failure has ~9 test samples — per-class metrics have high variance.')

# Confusion matrix
cm_ft = confusion_matrix(y_ft_test, y_ft_pred, labels=ft_classes)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_ft, display_labels=ft_classes)
disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=30)
ax.set_title(
    f'Failure-Type Confusion Matrix\n{best_ft_name}  |  test n={len(y_ft_test)}',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Save and verify failure-type model ─────────────────────
import joblib

MODEL_FT_PKL = MODEL_DIR / 'failure_type_model.pkl'

joblib.dump(best_ft_pipeline, MODEL_FT_PKL)
print(f'Failure-type model saved : {MODEL_FT_PKL}')
print(f'File size                : {MODEL_FT_PKL.stat().st_size / 1024:.1f} KB')

# Reload and verify
loaded_ft = joblib.load(MODEL_FT_PKL)
y_ft_pred_verify = loaded_ft.predict(X_ft_test)
assert (y_ft_pred_verify == y_ft_pred).all(), 'RELOAD CHECK FAILED'

print()
print('Reload verification      : PASS')
print(f'  Type                   : {type(loaded_ft).__name__}')
print(f'  Steps                  : {[s for s, _ in loaded_ft.steps]}')
print(f'  Prediction check       : all {len(y_ft_pred)} test predictions match')
print()
print('Section 7 complete.')
print(f'  model/best_model.pkl         — PRIMARY   binary classifier (unchanged)')
print(f'  model/failure_type_model.pkl — SECONDARY failure-type classifier')

---
## Section 8 — Saved Models: Status Check

Both models were saved in Sections 6 and 7.  
This section confirms both `.pkl` files exist and can be reloaded before running predictions.

| File | Purpose |
|------|---------|
| `model/best_model.pkl` | Primary binary classifier (No Failure vs Failure) |
| `model/failure_type_model.pkl` | Secondary multiclass classifier (failure type, called only when primary predicts Failure) |

In [ ]:
# ── Confirm both model files exist and reload cleanly ─────────
import joblib

MODEL_FT_PKL = MODEL_DIR / 'failure_type_model.pkl'

for label, path in [('Primary   (best_model.pkl)        ', MODEL_PKL),
                    ('Secondary (failure_type_model.pkl)', MODEL_FT_PKL)]:
    assert path.exists(), f'MISSING: {path}'
    pipe = joblib.load(path)
    print(f'{label}  {path.stat().st_size/1024:.1f} KB  '
          f'steps={[s for s,_ in pipe.steps]}  OK')

# Keep references for Section 9
primary_model   = joblib.load(MODEL_PKL)
secondary_model = joblib.load(MODEL_FT_PKL)

print()
print('Both models loaded. Ready for sample predictions (Section 9).')

---
## Section 9 — End-to-End Sample Predictions

Demonstrates the two-stage prediction logic used by the Flask API:
1. Feed the input to the **primary model** → binary prediction (0 = No Failure, 1 = Failure)
2. **Only if** the primary model predicts Failure → feed the same input to the **secondary model** → failure type

All predictions come from the loaded `.pkl` files. No results are pre-filled.

In [ ]:
# ── Sample inputs (hand-crafted, realistic sensor values) ─────
# Each dict must contain exactly the six feature columns in FEATURE_COLUMNS order.

sample_inputs = [
    {
        'label'                     : 'Normal operating conditions',
        'Type'                      : 'M',
        'Air temperature [K]'       : 300.0,
        'Process temperature [K]'   : 310.0,
        'Rotational speed [rpm]'    : 1500,
        'Torque [Nm]'               : 40.0,
        'Tool wear [min]'           : 50,
    },
    {
        'label'                     : 'High torque / worn tool (likely failure)',
        'Type'                      : 'L',
        'Air temperature [K]'       : 302.5,
        'Process temperature [K]'   : 311.5,
        'Rotational speed [rpm]'    : 1200,
        'Torque [Nm]'               : 68.0,
        'Tool wear [min]'           : 240,
    },
    {
        'label'                     : 'High speed / low torque',
        'Type'                      : 'H',
        'Air temperature [K]'       : 298.0,
        'Process temperature [K]'   : 308.5,
        'Rotational speed [rpm]'    : 2500,
        'Torque [Nm]'               : 10.0,
        'Tool wear [min]'           : 5,
    },
    {
        'label'                     : 'Moderate wear / moderate torque',
        'Type'                      : 'M',
        'Air temperature [K]'       : 299.8,
        'Process temperature [K]'   : 309.7,
        'Rotational speed [rpm]'    : 1450,
        'Torque [Nm]'               : 55.0,
        'Tool wear [min]'           : 200,
    },
]

print('End-to-End Sample Predictions')
print('=' * 70)

for idx, sample in enumerate(sample_inputs, 1):
    label  = sample.pop('label')
    # Build a single-row DataFrame in the exact feature column order
    row_df = pd.DataFrame([{col: sample[col] for col in FEATURE_COLUMNS}])

    # Stage 1: Primary binary prediction
    binary_pred  = primary_model.predict(row_df)[0]
    binary_proba = primary_model.predict_proba(row_df)[0]  # [p_no_fail, p_fail]
    failure_prob = binary_proba[1]

    print(f'\nSample {idx}: {label}')
    print(f'  Input          : Type={sample["Type"]}  '
          f'Torque={sample["Torque [Nm]"]} Nm  '
          f'Speed={sample["Rotational speed [rpm]"]} rpm  '
          f'Wear={sample["Tool wear [min]"]} min')
    print(f'  Primary result : {"FAILURE" if binary_pred==1 else "No Failure"}  '
          f'(failure probability: {failure_prob:.3f})')

    # Stage 2: Secondary failure-type prediction (only when primary == 1)
    if binary_pred == 1:
        ft_pred  = secondary_model.predict(row_df)[0]
        ft_proba = secondary_model.predict_proba(row_df)[0]
        ft_classes = secondary_model.classes_
        ft_conf  = ft_proba.max()
        print(f'  Failure type   : {ft_pred}  (confidence: {ft_conf:.3f})')
    else:
        print(f'  Failure type   : N/A  (secondary model not called — no failure predicted)')

    # Restore label for readability
    sample['label'] = label

print()
print('=' * 70)
print('All predictions produced by loaded .pkl models. No values are pre-filled.')

---
## End of Notebook

All results above were produced by running this notebook on the actual dataset.  
See `train_model.py` for the reproducible training script without visualisations.  
See `backend/app.py` for the Flask REST API that exposes these same two models.